# Step 00 — Resolve & freeze ENCODE accessions

Companion notebook for `workflow/scripts/00_resolve_accessions.py`. Explore
what ENCODEfetch resolved for the frozen pilot accession(s) in
`resources/accessions/pilot_accessions.txt`, and sanity-check it against
the paper's own Table S1.

Run `snakemake data/accessions/manifest.tsv` at least once before running
this notebook.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root(marker="pixi.toml"):
    # Jupyter starts a notebook's kernel with the notebook's own directory as
    # cwd, not wherever the server was launched from, so paths below can't
    # just assume cwd == repo root.
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} above {Path.cwd()}")


REPO_ROOT = find_repo_root()

In [ ]:
config = yaml.safe_load((REPO_ROOT / "config/config.yaml").read_text())
data = REPO_ROOT / config["paths"]["data"]
manifest_path = data / "accessions" / "manifest.tsv"
metadata_path = data / "accessions" / "metadata.jsonl"
samplesheet_path = data / "accessions" / "samples.csv"

## The frozen accession(s) that produced this manifest

In [ ]:
print((REPO_ROOT / config["accessions"]["pilot_file"]).read_text())

## Resolved manifest

One row per FASTQ file, case and matched control together.

In [ ]:
manifest = pd.read_csv(manifest_path, sep="\t")
manifest[
    [
        "experiment_accession",
        "is_control",
        "biosample_term_name",
        "target_label",
        "file_accession",
        "file_accession_r2",
        "file_size",
        "file_size_r2",
        "md5sum",
        "md5sum_r2",
    ]
]

## Sanity checks

These mirror what the paper's Table S1 records for this pilot pair — a
CTCF experiment and its matched Control (input) experiment, each library
split into two paired-end FASTQ files.

In [ ]:
assert manifest["biosample_term_name"].nunique() == 1, "Expected a single biosample for the pilot"
print("Biosample:", manifest["biosample_term_name"].iloc[0])

In [ ]:
assert set(manifest["target_label"].dropna().unique()) <= {"CTCF"}, manifest["target_label"].unique()
print("Experiments:", sorted(manifest["experiment_accession"].unique()))

## Total download size for this pilot

In [ ]:
total_bytes = manifest[["file_size", "file_size_r2"]].sum().sum()
print(f"{total_bytes / 1e9:.2f} GB across {len(manifest) * 2} FASTQ files")

## Snakemake samplesheet ENCODEfetch generated

In [ ]:
pd.read_csv(samplesheet_path)

## Raw metadata record for one file, for anything the manifest trims

In [ ]:
with metadata_path.open() as handle:
    first_record = json.loads(next(handle))
first_record